In [247]:
import torch
import torch.nn.functional as F
import pandas as pd
import random
import torch.nn as nn
torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [248]:
df = pd.read_csv("dataset.csv")
df = df.drop_duplicates(subset="FEN")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
X_fen = df["FEN"][:100000]
Y = df["EVAL"][:100000]


In [249]:
def get_stm(fen):
    fen_parts = fen.split(' ')
    return 0 if (fen_parts[1] == 'w') else 1

def fen_to_halfkp(fen):
    w, b = [], []
    fen_parts = fen.split(' ')
    piece_to_int = {'K':0, 'Q':1, 'R':2, 'B':3, 'N':4, 'P':5,
                    'k':6, 'q':7, 'r':8, 'b':9, 'n':10, 'p':11}
    b_king, w_king = None, None

    row, col = 7, 0
    for c in fen_parts[0]:
        if c == '/':
            row -= 1
            col = 0
        elif c.isdigit():
            col += int(c)
        else:
            if c == 'K':
                w_king = row * 8 + col
            elif c == 'k':
                b_king = (7 - row) * 8 + col
            col += 1

    row, col = 7, 0
    for c in fen_parts[0]:
        if c == '/':
            row -= 1
            col = 0
        elif c.isdigit():
            col += int(c)
        else:
            if c == 'K' or c == 'k':
                col += 1
                continue
            piece_info = piece_to_int[c]
            piece_color = piece_info // 6
            piece_type = piece_info % 6 - 1 
            sq_w = row * 8 + col
            sq_b = (7 - row) * 8 + col
            final_idx_w = sq_w + piece_color * 64 + piece_type * 64 * 2 + w_king * 64 * 2 * 5
            final_idx_b = sq_b + (1 - piece_color) * 64 + piece_type * 64 * 2 + b_king * 64 * 2 * 5
            w.append(final_idx_w)
            b.append(final_idx_b)
            col += 1
    pos = [w,b]
    return pos


In [250]:
X = X_fen.map(fen_to_halfkp)
Y = torch.tensor(Y.to_list()).float().to(device)
stm = torch.tensor(X_fen.map(get_stm).to_list()).float().unsqueeze(1).to(device)


In [251]:
n1 = int(0.92 * len(X))
n2 = int(0.96 * len(X))
X_train, Y_train, stm_train = X[:n1], Y[:n1], stm[:n1]
X_dev,   Y_dev,   stm_dev   = X[n1:n2], Y[n1:n2], stm[n1:n2]
X_test,  Y_test,  stm_test  = X[n2:], Y[n2:], stm[n2:]


In [252]:
INPUT_SIZE = 40960
M = 256
N = 32
K = 1

class PositionEvaluator(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = torch.nn.Linear(INPUT_SIZE, M)
        self.ln2 = torch.nn.Linear(2 * M, N)
        self.ln3 = torch.nn.Linear(N, N)
        self.ln4 = torch.nn.Linear(N,K)


    def forward(self, x, color):
        w = self.ln1(x[:, 0])
        b = self.ln1(x[:, 1])
        accumulator = (1 - color) * torch.cat([w, b], dim=1) + color * torch.cat([b, w], dim=1)
        h1 = torch.clamp(accumulator, 0.0, 1.0)
        h2 = torch.clamp(self.ln2(h1), 0.0, 1.0)
        h3 = torch.clamp(self.ln3(h2),0.0,1.1)
        out = self.ln4(h3)
        return out


In [253]:
model = PositionEvaluator().to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3  )
loss_fn = torch.nn.MSELoss()

In [255]:
def build_x(X_src, idx):
    X_batch = torch.zeros(len(idx), 2, INPUT_SIZE)
    for cur, i in enumerate(idx):
        X_i = X_src.iloc[i]
        w_idx = X_i[0]
        b_idx = X_i[1]
        for w in w_idx:
            X_batch[cur][0][w] = 1
        for b in b_idx:
            X_batch[cur][1][b] = 1
    return X_batch.to(device)

In [260]:
epochs = 50000
batch_size = 32
dev_batch_size = X_dev.shape[0]
scalling = 400

dev_idx = torch.randint(0, X_dev.shape[0], (dev_batch_size,))
X_dev_batch = build_x(X_dev, dev_idx.tolist())
stm_dev_batch = stm_dev[dev_idx]
dev_target = torch.sigmoid(Y_dev[dev_idx].unsqueeze(1) / scalling)

for i in range(epochs):
    idx = torch.randint(0, X_train.shape[0], (batch_size,))
    X_batch = build_x(X_train, idx.tolist())
    stm_batch = stm_train[idx]
    Y_batch = torch.sigmoid(Y_train[idx].unsqueeze(1) / scalling)
    output = torch.sigmoid(model(X_batch, stm_batch) / scalling)
    loss = loss_fn(output, Y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % 1000 == 0:
        with torch.no_grad():
            dev_pred = torch.sigmoid(model(X_dev_batch, stm_dev_batch) / scalling)
            dev_loss = loss_fn(dev_pred, dev_target)
            print(f"{i}/{epochs} --> train {loss.item():.4f}  dev {dev_loss.item():.4f}")


0/50000 --> train 0.0031  dev 0.0152
1000/50000 --> train 0.0029  dev 0.0149
2000/50000 --> train 0.0042  dev 0.0153
3000/50000 --> train 0.0022  dev 0.0158
4000/50000 --> train 0.0021  dev 0.0156
5000/50000 --> train 0.0020  dev 0.0156
6000/50000 --> train 0.0018  dev 0.0153
7000/50000 --> train 0.0027  dev 0.0152
8000/50000 --> train 0.0029  dev 0.0148
9000/50000 --> train 0.0039  dev 0.0148
10000/50000 --> train 0.0029  dev 0.0152
11000/50000 --> train 0.0025  dev 0.0146
12000/50000 --> train 0.0019  dev 0.0148
13000/50000 --> train 0.0017  dev 0.0148
14000/50000 --> train 0.0022  dev 0.0146
15000/50000 --> train 0.0023  dev 0.0147
16000/50000 --> train 0.0013  dev 0.0142
17000/50000 --> train 0.0014  dev 0.0141
18000/50000 --> train 0.0013  dev 0.0144
19000/50000 --> train 0.0019  dev 0.0145
20000/50000 --> train 0.0041  dev 0.0147
21000/50000 --> train 0.0025  dev 0.0143
22000/50000 --> train 0.0032  dev 0.0146
23000/50000 --> train 0.0012  dev 0.0144
24000/50000 --> train 0.0009 

In [261]:
with torch.no_grad():
    X_test_batch = build_x(X_test, list(range(X_test.shape[0])))
    test_target = torch.sigmoid(Y_test.unsqueeze(1) / scalling)
    output = torch.sigmoid(model(X_test_batch, stm_test) / scalling)
    print(loss_fn(output, test_target))

tensor(0.0157, device='mps:0')


In [262]:
with torch.no_grad():
    X_test_batch = build_x(X_test, list(range(X_test.shape[0])))
    outputs = model(X_test_batch,stm_test).view(-1)
    print(outputs[:10])
    print(Y_test[:10])

tensor([ -17.9841,  361.3112,    8.6638,  231.1103,   15.0752,   69.6244,
        -374.2023,  737.6477,  737.6477,   -8.1560], device='mps:0')
tensor([  -4.,  616., -127.,  437.,   14.,   69., -538., 1270.,  581.,   -5.],
       device='mps:0')


In [263]:
cnt = 0
for i in range(outputs.shape[0]):
    if(outputs[i].item() < 0 and Y_test[i] > 0):
        cnt += 1
    elif(outputs[i].item() > 0 and Y_test[i] < 0):
        cnt += 1
print(cnt)
print(cnt/output.shape[0])

845
0.21125
